# <center> Homework 109

In [2]:
from importlib import reload
import finetuning
import tensorflow as tf
import numpy as np

2026-01-15 23:27:51.948670: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-15 23:27:52.889444: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-15 23:27:56.442584: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Task 5

In [3]:
np.random.seed(13)
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42, test_size=0.2, shuffle=True) 
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, random_state=42, test_size=0.1, shuffle=True) 

In [20]:
def build_model(hp):
    n_neurons = hp.Int('n_neurons', 3, 5)
    n_hidden = hp.Choice('n_hidden', [3, 4])
    lr = hp.Float('lr', 0.001, 0.1, step=0.05)

    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(X_train.shape[1:]))
    model.add(tf.keras.layers.Normalization())

    # print(type(n_neurons))
    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons,
                                        activation='relu',
                                        kernel_initializer='he_normal'))

    model.add(tf.keras.layers.Dense(1))
    optimizer = tf.keras.optimizers.Adam(lr)
    model.compile(loss='mse',
                  optimizer=optimizer,
                  metrics=['RootMeanSquaredError'])
    
    return model

In [22]:
reload(finetuning)
from finetuning import GridSearch

tuner = GridSearch(build_model, 'val_RootMeanSquaredError',
                   directory='gridsearach', project_name='test1')

tuner.search(X_train, y_train, epochs=5, validation_data=(X_val, y_val))


Trail 1/12
n_neurons                      3
n_hidden                       3
lr                             0.001

Epoch 1/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - RootMeanSquaredError: 2.1805 - loss: 4.7548 - val_RootMeanSquaredError: 2.0444 - val_loss: 4.1796
Epoch 2/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - RootMeanSquaredError: 1.8422 - loss: 3.3938 - val_RootMeanSquaredError: 1.7420 - val_loss: 3.0344
Epoch 3/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - RootMeanSquaredError: 1.5763 - loss: 2.4848 - val_RootMeanSquaredError: 1.5127 - val_loss: 2.2883
Epoch 4/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - RootMeanSquaredError: 1.3842 - loss: 1.9159 - val_RootMeanSquaredError: 1.3545 - val_loss: 1.8348
Epoch 5/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - RootMeanSquaredError: 1.2613 - loss: 1.5908 - val_RootMeanSquaredError: 1.2597 - val_loss: 1.5868

Trail 2/12
n_neurons                      3
n_hidden                       3
lr                             0.0505

Epoch 1/5
46

In [23]:
best_model = tuner.get_best_models(0)[0]
best_model.evaluate(X_test, y_test)

129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - RootMeanSquaredError: 1.0827 - loss: 1.1722


[1.1722323894500732, 1.082696795463562]

In [24]:
import pandas as pd

for params in tuner.get_best_hyperparameters(3):
    print(pd.Series(params))

n_neurons    3.000
n_hidden     3.000
lr           0.001
dtype: float64
n_neurons    3.000
n_hidden     4.000
lr           0.001
dtype: float64
n_neurons    4.000
n_hidden     3.000
lr           0.001
dtype: float64


In [28]:
reload(finetuning)
from finetuning import GridSearch

tuner = GridSearch(build_model, 'val_RootMeanSquaredError', max_trials=3,
                   directory='gridsearach', project_name='test2')

tuner.search(X_train, y_train, epochs=5, validation_data=(X_val, y_val))

tuner.search(X_train, y_train, epochs=3, validation_data=(X_val, y_val))


Trail 1/3
n_neurons                      3
n_hidden                       3
lr                             0.001

Epoch 1/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - RootMeanSquaredError: 157.3528 - loss: 24759.9023 - val_RootMeanSquaredError: 32.2266 - val_loss: 1038.5505
Epoch 2/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - RootMeanSquaredError: 19.4456 - loss: 378.1326 - val_RootMeanSquaredError: 9.2937 - val_loss: 86.3733
Epoch 3/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - RootMeanSquaredError: 6.5144 - loss: 42.4372 - val_RootMeanSquaredError: 4.3865 - val_loss: 19.2416
Epoch 4/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - RootMeanSquaredError: 3.6217 - loss: 13.1168 - val_RootMeanSquaredError: 3.0679 - val_loss: 9.4122
Epoch 5/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - RootMeanSquaredError: 2.8676 - loss: 8.2231 - val_RootMeanSquaredError: 2.6948 - val_loss: 7.2619

Trail 2/3
n_neurons                      3
n_hidden                       3
lr                             0.050

## Task 6

In [33]:
import tf_model
reload(tf_model)
from tf_model import Dense, Input, Normalization, Sequential, SGD

def build_model(hp):
    n_neurons = hp.Int('n_neurons', 3, 5)
    n_hidden = hp.Choice('n_hidden', [3, 4])
    lr = hp.Float('lr', 0.001, 0.1, step=0.05)

    model = Sequential()
    model.add(Input(X_train.shape[1:]))

    norm = Normalization()
    norm.adapt(X_train)
    model.add(norm)

    for _ in range(n_hidden):
        model.add(Dense(n_neurons,
                        activation='relu',
                        kernel_initializer='he_normal'))

    model.add(Dense(1))
    optimizer = SGD(lr)
    model.compile(loss='mse',
                  optimizer=optimizer,
                  metrics=['rmse'])
    
    return model

In [34]:
reload(finetuning)
from finetuning import GridSearch

tuner = GridSearch(build_model, 'val_rmse',
                   directory='gridsearach', project_name='test3')

tuner.search(X_train, y_train, epochs=5, validation_data=(X_val, y_val))


Trail 1/12
n_neurons                      3
n_hidden                       3
lr                             0.001

Epoch 1/5


  0%|          | 0/465 [00:00<?, ?it/s]

100%|██████████| 465/465 [00:01<00:00, 407.11it/s]


    - loss: 3.7232025096069195 - rmse: 2.365825150832799
    - val_loss: 2.8662234989962 - val_rmse: 1.692992468676751
Learning Rate: 0.001
Epoch 2/5


100%|██████████| 465/465 [00:01<00:00, 444.31it/s]


...

    - loss: 0.4518613125345332 - rmse: 0.6128745278894535
    - val_loss: 0.45428641823280563 - val_rmse: 0.6740077286150402
Learning Rate: 0.1
Epoch 4/5


100%|██████████| 465/465 [00:01<00:00, 412.23it/s]


    - loss: 0.4552466213023122 - rmse: 0.42721650855401594
    - val_loss: 0.44865944075113395 - val_rmse: 0.6698204541152307
Learning Rate: 0.1
Epoch 5/5


100%|██████████| 465/465 [00:01<00:00, 398.24it/s]

    - loss: 0.4368645517816782 - rmse: 0.5688661890124568
    - val_loss: 0.49002000147649577 - val_rmse: 0.7000142866231344
Learning Rate: 0.1
